# Lecture 24 Worksheet: Activations, Softmax, Stable Loss, Adam
**NAVTTC / NIAI AI/ML Trainer Cohort** — works in Google Colab. Fill every `___` blank and run top to bottom; check lines print `OK`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow.keras import layers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
np.set_printoptions(precision=4, suppress=True)

## Task 1: Two linear layers collapse into one
Layer 1: `a1 = 3*x + 1`. Layer 2: `a2 = 0.5*a1 + 2`. Show that one layer `w*x + b` gives the same answers. Fill in `w` and `b`.

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
a1 = 3*x + 1
a2 = 0.5*a1 + 2
w = ___          # w2 * w1
b = ___          # w2 * b1 + b2
print(a2, w*x + b)
assert np.allclose(a2, w*x + b); print("OK, depth without an activation adds nothing")

## Task 2: Softmax by hand
Three vehicles with scores `z = [2.5, 1.0, -1.0]` (bike, scooter, van). Write softmax and check it adds up to 1.

In [ ]:
def softmax(z):
    e = np.exp(___)
    return e / ___

z = np.array([2.5, 1.0, -1.0])
a = softmax(z)
print(a, a.sum())
assert np.isclose(a.sum(), 1) and np.allclose(a, [0.798, 0.178, 0.024], atol=0.001); print("OK")
print("loss if the true class is scooter:", -np.log(a[1]))

## Task 3: N = 2 softmax is a sigmoid
For `z = [2.5, 1.0]` show that softmax of the first score equals `sigmoid(z1 - z2)`.

In [ ]:
sigmoid = lambda t: 1/(1+np.exp(-t))
two = softmax(np.array([2.5, 1.0]))
print(two[0], sigmoid(___))
assert np.isclose(two[0], sigmoid(2.5 - 1.0)); print("OK")

## Task 4: The round-off problem
In 32-bit numbers, sigmoid of a big score becomes exactly 1. Then `1 - a` is 0 and `log(0)` is infinite.

In [ ]:
z = np.float32(40)
a = np.float32(1) / (np.float32(1) + np.exp(-z))
print("a =", a, " 1 - a =", np.float32(1) - a)
naive_loss = -np.log(np.float32(1) - a)         # true label is 0
print("naive loss:", naive_loss, " correct loss: ", ___)    # the correct loss for y=0 is z itself, 40
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
print("Keras from_logits:", bce(tf.constant([[0.0]]), tf.constant([[40.0]])).numpy())
assert np.isinf(naive_loss) and abs(bce(tf.constant([[0.0]]), tf.constant([[40.0]])).numpy() - 40) < 1e-3; print("OK")

In [ ]:
z = np.array([1000., 1001., 1002.], dtype=np.float32)
with np.errstate(all="ignore"):
    print("naive:", np.exp(z) / np.exp(z).sum())
s = np.exp(z - z.max())          # trick: subtract the biggest score first
print("stable:", s / s.sum())

## Task 5: Safe Keras version on digits
Same 10-class network twice. The first has softmax in the last layer; the second has a linear last layer and `from_logits=True`. Fill the blanks.

In [ ]:
d = load_digits()
X = (d.data / 16.0).astype("float32"); y = d.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)

def make(last_activation, from_logits):
    tf.keras.utils.set_random_seed(1)
    m = tf.keras.Sequential([tf.keras.Input(shape=(64,)),
        layers.Dense(25, activation="relu"), layers.Dense(15, activation="relu"),
        layers.Dense(10, activation=last_activation)])
    m.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=from_logits),
              optimizer=tf.keras.optimizers.Adam(0.01), metrics=["accuracy"])
    return m

m1 = make("softmax", False);  m1.fit(Xtr, ytr, epochs=40, verbose=0)
m2 = make("___", ___);        m2.fit(Xtr, ytr, epochs=40, verbose=0)
print("softmax in layer :", m1.evaluate(Xte, yte, verbose=0)[1])
print("from_logits      :", m2.evaluate(Xte, yte, verbose=0)[1])

logits = m2.predict(Xte, verbose=0)
probs = ___(logits)                      # turn raw scores into chances
print("row sums:", probs.sum(axis=1)[:3], " same class as argmax of logits:", (probs.argmax(1) == logits.argmax(1)).all())
assert np.allclose(probs.sum(axis=1), 1, atol=1e-5) and m2.evaluate(Xte, yte, verbose=0)[1] > 0.9; print("OK")

## Task 6: Plain gradient descent vs Adam on a narrow valley
Loss `J = 0.5*(w1**2 + 30*w2**2)`: very gentle in `w1`, very steep in `w2`. Both start at `(-4, 1.5)`. Count the steps to reach `J < 0.01`.

In [ ]:
A = np.array([1.0, 30.0])
J = lambda w: 0.5*np.sum(A*w*w)
grad = lambda w: ___                     # derivative of 0.5*A*w^2 is A*w

def gd(alpha, steps=300):
    w = np.array([-4.0, 1.5])
    for t in range(steps):
        if J(w) < 0.01: return t
        w = w - alpha*grad(w)
    return None

def adam(alpha, steps=300, b1=0.9, b2=0.999, eps=1e-8):
    w = np.array([-4.0, 1.5]); m = np.zeros(2); v = np.zeros(2)
    for t in range(1, steps+1):
        if J(w) < 0.01: return t-1
        g = grad(w); m = b1*m + (1-b1)*g; v = b2*v + (1-b2)*g*g
        w = w - alpha*(m/(1-b1**t))/(np.sqrt(v/(1-b2**t)) + eps)
    return None

print("plain GD alpha 0.065:", gd(0.065), " alpha 0.07:", gd(0.07), " Adam alpha 0.2:", adam(0.2))
assert adam(0.2) < gd(0.065); print("OK")

## Task 7: Your own project
In two sentences: what is your target `y`? Which output-layer activation and loss does it need (see slide 3)? Would you use `from_logits=True`?

In [ ]:
my_answer = "___"
print(my_answer)

## Wrap-up
- No activation between layers = a straight line, however deep.
- Output layer: sigmoid (yes/no), linear (number), softmax (many classes).
- Safe numbers: linear last layer + `from_logits=True`; apply sigmoid or softmax after `predict`.
- Adam gives each weight its own step size; learning rate still needs a sensible start.
- Next: Lecture 25, how to tell whether a model has high bias or high variance.